# 🌌 Antigravity AI Video Detector for Instagram
이 노트북은 인스타그램 동영상 링크를 분석하여 C2PA, VideoSeal, Google SynthID 기술을 통해 AI 생성 여부를 판독합니다.

In [ ]:
# 필요한 라이브러리 설치 (필요한 경우 주석 해제 후 실행)
# !pip install instaloader c2pa-python videoseal torch torchvision opencv-python pillow matplotlib

In [ ]:
import os
import sys
import torch
import json
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# insta-ai-checker 경로 추가
parent_dir = os.path.abspath("..")
checker_dir = os.path.join(parent_dir, "insta-ai-checker")
sys.path.append(checker_dir)

from utils.downloader import download_instagram_media
from utils.c2pa_checker import check_c2pa
from utils.synthid_checker import check_synthid

print("✅ 환경 설정 및 유틸리티 로드 완료")

## 1. VideoSeal 탐지 모듈 설정
Facebook Research의 VideoSeal 모델을 사용하여 워터마크를 탐지합니다.

In [ ]:
def check_videoseal(file_path):
    """
    VideoSeal 워터마크를 탐지합니다.
    """
    try:
        import videoseal
        import torch
        import cv2
        import os
        import requests
        from pathlib import Path
        from videoseal.utils.cfg import setup_model
        from omegaconf import OmegaConf
        
        # 모델 설정 파일의 절대 경로 찾기
        base_path = os.path.dirname(videoseal.__file__)
        config_path = Path(base_path) / 'cards' / 'videoseal_1.0.yaml'
        if not config_path.exists():
            config_path = Path(base_path) / 'cards' / 'videoseal_0.0.yaml'
            
        if not config_path.exists():
            return False, "VideoSeal 설정 파일(.yaml)을 찾을 수 없습니다."

        config = OmegaConf.load(config_path)
        
        # 1. Attenuation 패치: JND 로직 우회
        if 'args' in config:
            config.args.attenuation = "none"
            
        # 2. 체크포인트 다운로드: URL일 경우 로컬로 다운로드
        ckpt_url = config.checkpoint_path
        if str(ckpt_url).startswith("http"):
            ckpt_filename = os.path.basename(ckpt_url)
            local_ckpt_path = Path(base_path) / "cards" / ckpt_filename
            
            if not local_ckpt_path.exists():
                print(f"📥 가중치 다운로드 중: {ckpt_url}")
                response = requests.get(ckpt_url, stream=True)
                response.raise_for_status()
                with open(local_ckpt_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print("✅ 다운로드 완료")
            ckpt_path = local_ckpt_path
        else:
            ckpt_path = Path(ckpt_url)

        # 3. 모델 초기화
        model = setup_model(config, ckpt_path)
            
        if model is None:
            return False, "VideoSeal 모델을 초기화할 수 없습니다."
            
        model.eval()
        if torch.cuda.is_available():
            model = model.cuda()

        cap = cv2.VideoCapture(file_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # 30개 프레임으로 대폭 상향
        check_points = np.linspace(0, total_frames - 1, 30, dtype=int)
        
        detected_count = 0
        total_score = 0
        
        for idx in check_points:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: continue
            
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_tensor = torch.from_numpy(frame_rgb).permute(2, 0, 1).float() / 255.0
            img_tensor = img_tensor.unsqueeze(0)
            
            if torch.cuda.is_available():
                img_tensor = img_tensor.cuda()
            
            with torch.no_grad():
                outputs = model.detect(img_tensor)
                if isinstance(outputs, torch.Tensor):
                    score = outputs.mean().item()
                    if score < 0.1 or score > 1.0:
                        score = torch.sigmoid(outputs).mean().item()
                else:
                    score = 0
                
                total_score += score
                if score > 0.5: 
                    detected_count += 1
        
        cap.release()
        
        avg_score = total_score / len(check_points) if len(check_points) > 0 else 0
        # 30개 중 6개(20%) 이상 탐지 시 최종 탐지로 판정
        is_found = detected_count >= 6 
        
        msg = f"### [VideoSeal 분석 결과]\n"
        msg += f"- **최종 판정:** {'AI 워터마크 탐지' if is_found else '워터마크 미감지'}\n"
        msg += f"- **평균 탐지 점수:** {avg_score:.4f}\n"
        msg += f"- **탐지된 프레임:** {detected_count}/30\n"
        msg += f"- **기술적 근거:** Meta의 VideoSeal 기술을 이용해 30개의 샘플 프레임 내에 삽입된 비가시적 워터마크 패턴을 분석했습니다."
        
        return is_found, msg
    except Exception as e:
        return False, f"VideoSeal 분석 중 에러 발생: {str(e)}"

## 2. 통합 분석 실행
인스타그램 URL을 입력하여 모든 검사 도구를 실행합니다.

In [ ]:
def run_antigravity_detection(url):
    print(f"🚀 분석 시작: {url}")
    
    # 1. 다운로드
    target_dir = os.path.join(checker_dir, "temp")
    file_path, media_type = download_instagram_media(url, target_dir)
    
    if not file_path:
        print(f"❌ 다운로드 실패: {media_type}")
        return
    
    print(f"📦 미디어 다운로드 완료: {os.path.basename(file_path)} ({media_type})")
    
    results = {}
    
    # 2. C2PA 검사
    print("🔍 1/3 C2PA 분석 중...")
    c2pa_found, c2pa_msg = check_c2pa(file_path)
    results['C2PA'] = {"found": c2pa_found, "message": c2pa_msg}
    
    # 3. SynthID 검사 (reverse-SynthID)
    print("🔍 2/3 SynthID 분석 중...")
    synth_found, synth_msg, synth_viz = check_synthid(file_path, media_type)
    results['SynthID'] = {"found": synth_found, "message": synth_msg, "viz": synth_viz}
    
    # 4. VideoSeal 검사
    print("🔍 3/3 VideoSeal 분석 중...")
    vseal_found, vseal_msg = check_videoseal(file_path)
    results['VideoSeal'] = {"found": vseal_found, "message": vseal_msg}
    
    # 결과 출력
    print("\n" + "="*50)
    print("📊 ANTIGRAVITY DETECTION REPORT")
    print("="*50)
    
    print(f"[C2PA Result]: {'✅ 탐지됨' if c2pa_found else '❌ 미탐지'}")
    print(f"상세: {c2pa_msg}\n")
    
    print(f"[SynthID Result]: {'✅ 탐지됨' if synth_found else '❌ 미탐지'}")
    print(f"상세: {synth_msg}\n")
    
    print(f"[VideoSeal Result]: {'✅ 탐지됨' if vseal_found else '❌ 미탐지'}")
    print(f"상세: {vseal_msg}\n")
    
    if synth_viz and os.path.exists(synth_viz):
        plt.figure(figsize=(10, 5))
        img = Image.open(synth_viz)
        plt.imshow(img)
        plt.axis('off')
        plt.title("SynthID Spectral Analysis")
        plt.show()

    return results

## 3. 실행
아래 변수에 인스타그램 동영상(Reels 또는 게시물) 주소를 입력하세요.

In [ ]:
INSTAGRAM_URL = "https://www.instagram.com/reels/DF5hFq_S7Y_/" # 예시 URL
run_antigravity_detection(INSTAGRAM_URL)